In [1]:
import pandas as pd
df = pd.read_csv("/content/global_ads_performance_dataset.csv")
df.head()

,date,platform,campaign_type,industry,country,impressions,clicks,CTR,CPC,ad_spend,conversions,CPA,revenue,ROAS
0,2024-01-21,Google Ads,Search,Fintech,UAE,59886,2113,0.0353,1.26,2662.38,159,16.74,4803.43,1.80
1,2024-01-22,TikTok Ads,Search,EdTech,UK,135608,5220,0.0385,1.18,6159.60,411,14.99,64126.68,10.41
2,2024-06-15,TikTok Ads,Video,Healthcare,USA,92313,5991,0.0649,0.85,5092.35,267,19.07,10489.07,2.06
3,2024-01-02,TikTok Ads,Shopping,SaaS,Germany,83953,5935,0.0707,1.32,7834.20,296,26.47,50505.07,6.45
4,2024-02-22,TikTok Ads,Search,Healthcare,UK,91807,4489,0.0489,1.93,8663.77,107,80.97,3369.53,0.39


# DATASET OVERVIEW

In [2]:
df.info()
df.describe()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1800 entries, 0 to 1799
Data columns (total 14 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   date           1800 non-null   object 
 1   platform       1800 non-null   object 
 2   campaign_type  1800 non-null   object 
 3   industry       1800 non-null   object 
 4   country        1800 non-null   object 
 5   impressions    1800 non-null   int64  
 6   clicks         1800 non-null   int64  
 7   CTR            1800 non-null   float64
 8   CPC            1800 non-null   float64
 9   ad_spend       1800 non-null   float64
 10  conversions    1800 non-null   int64  
 11  CPA            1800 non-null   float64
 12  revenue        1800 non-null   float64
 13  ROAS           1800 non-null   float64
dtypes: float64(6), int64(3), object(5)
memory usage: 197.0+ KB


,impressions,clicks,CTR,CPC,ad_spend,conversions,CPA,revenue,ROAS
count,1800.000000,1800.000000,1800.000000,1800.000000,1800.000000,1800.000000,1800.000000,1800.000000,1800.000000
mean,102919.018889,3962.675556,0.038427,1.572756,6171.527272,181.562222,46.608961,30101.850450,6.450367
std,55740.900690,2941.858037,0.017082,0.800872,5776.996958,171.424239,41.185556,34560.032941,6.590986
min,5059.000000,91.000000,0.008900,0.280000,58.000000,2.000000,4.800000,142.690000,0.130000
25%,54948.000000,1678.000000,0.025400,0.950000,1966.587500,59.000000,20.202500,7275.757500,2.170000
50%,103653.000000,3318.000000,0.035550,1.460000,4393.860000,130.000000,33.375000,18362.965000,4.295000
75%,150470.250000,5628.000000,0.049800,2.050000,8455.830000,252.250000,56.812500,38963.385000,8.212500
max,199650.000000,16660.000000,0.095600,3.950000,38453.320000,1151.000000,335.860000,295028.260000,49.000000


+ platform: Google Ads / Meta / TikTok
+ campaign_type: loại chiến dịch
+ industry: ngành
+ impressions: só quảng cáo hiển thị
+ CTR: tỷ lệ click
+ CPC: chi phí mỗi click
+ ad_spend: chi phí quảng cáo
+ CPA: cost per acquisition
+ ROAS: return on ad spend

# DATA CLEANING

In [4]:
# check missing value
df.isnull().sum()
df = df.dropna()
# check duplicates
df.duplicated().sum()
df = df.drop_duplicates()
# fix date -> datetime
df["date"] = pd.to_datetime(df["date"])
df.dtypes


,0
date,datetime64[ns]
platform,object
campaign_type,object
industry,object
country,object
impressions,int64
clicks,int64
CTR,float64
CPC,float64
ad_spend,float64


# KIỂM TRA OUTLIERS (DETECT OUTLIERS)

In [5]:
# Các cột cần kiểm tra outliers
cols = ["CTR", "CPC", "CPA", "ROAS", "ad_spend", "revenue"]

In [8]:
# Dùng IQR để xử lí outliers
variables = ['impressions','clicks','CTR','CPC','ad_spend','conversions','CPA','revenue','ROAS']

for var in variables:

    Q1 = df[var].quantile(0.25)
    Q3 = df[var].quantile(0.75)

    IQR = Q3 - Q1

    lower = Q1 - 1.5 * IQR
    upper = Q3 + 1.5 * IQR

    outliers = df[(df[var] < lower) | (df[var] > upper)]

    print(f"{var}: {len(outliers)} outliers")

impressions: 0 outliers
clicks: 42 outliers
CTR: 12 outliers
CPC: 7 outliers
ad_spend: 90 outliers
conversions: 85 outliers
CPA: 140 outliers
revenue: 127 outliers
ROAS: 123 outliers


# VALIDATE LOGIC CỦA DATA

In [9]:
# Check CTR
df["CTR_check"] = df["clicks"] / df["impressions"]

In [10]:
df[["CTR","CTR_check"]].head()

,CTR,CTR_check
0,0.0353,0.035284
1,0.0385,0.038493
2,0.0649,0.064899
3,0.0707,0.070694
4,0.0489,0.048896


# FEATURE ENGINEERING

In [11]:
# Tạo thêm conversion rate
df["conversion_rate"] = df["conversions"] / df["clicks"]

In [12]:
# Tạo thêm ROI (lợi nhuận)
df["ROI"] = df["revenue"] / df["ad_spend"]

In [13]:
df.groupby("platform")["ROAS"].mean()

,ROAS
platform,
Google Ads,4.113028
Meta Ads,6.915730
TikTok Ads,9.538600


In [14]:
df.groupby("platform")["CTR"].mean()

,CTR
platform,
Google Ads,0.039856
Meta Ads,0.024983
TikTok Ads,0.054963


In [15]:
df.groupby("platform")["revenue"].sum()

,revenue
platform,
Google Ads,22033744.95
Meta Ads,11926045.79
TikTok Ads,20223540.07


# EXPORT DATASET

In [16]:
df.to_csv("global_ads_cleaned.csv", index=False)

In [17]:
from google.colab import files
files.download("global_ads_cleaned.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>